____
# Add editing flag and distance to coast

In [1]:
import numpy as np
import pandas as pd
import xarray as xr

import matplotlib.pyplot as plt

import os
from glob import glob

from cstes import swot_dir, drifters_dir, get_proj, lonlat2xy, zarr_dir
from swot import browse_swot_250

import cartopy.crs as ccrs
import cartopy.feature as cfeature
import cartopy.geodesic as cgeo
crs = ccrs.PlateCarree()

import cartopy.geodesic as geod
import cartopy.crs as ccrs
import cartopy.feature as cfeature

import dask.dataframe as dd

import pyproj
from pyproj import Geod

from rasterio.transform import Affine

import pynsitu as pyn

_____________
# On datarmor

In [ ]:
p = 15
if True:
    from dask.distributed import Client
    from dask_jobqueue import PBSCluster

    # cluster = PBSCluster(cores=56, processes=28, walltime='04:00:00')
    # cluster = PBSCluster(cores=7, processes=7, walltime='04:00:00')
    cluster = PBSCluster(cores=p, processes=p, walltime="04:00:00")
    w = cluster.scale(jobs=4)
else:
    from dask.distributed import Client, LocalCluster

    cluster = LocalCluster()

client = Client(cluster)
client

In [2]:
cluster.close()

NameError: name 'cluster' is not defined

_______
# Data

In [3]:
df = (pd.read_csv(os.path.join(zarr_dir, 'drifters_'+drifters_sources.replace('.nc', '.csv')), dtype={'drifter_id': 'str', 'pass_number':'int'})
     ).set_index('row_number')[['longitude', 'latitude', 'pass_number', 'cycle_number']]
pass_number = df.pass_number.unique()
for swath in pass_number : 
    cycle_number = df.where(df.pass_number==swath).dropna().cycle_number.unique()
    for cycle in cycle_number:
        df.where(df.pass_number==swath).where(df.cycle_number==cycle).dropna().to_csv(os.path.join(zarr_dir, 'coloc_pass_cycle', f'{int(swath)}_{int(cycle)}.csv'))
        print(swath, cycle)
                                                                                      

NameError: name 'zarr_dir' is not defined

In [3]:
dfs = browse_swot_250().reset_index()
drifters_sources = 'all_med_variational_10min_v0.nc'
#drifters_sources = 'all_med_lowess_10min_v0.nc'
#df = (dd.read_csv(os.path.join(zarr_dir, 'drifters_'+drifters_sources.replace('.nc', '.csv')), dtype={'drifter_id': 'str', 'pass_number':'int'})
#     ).set_index('row_number')[['longitude', 'latitude', 'pass_number', 'cycle_number']]

files = sorted(glob(os.path.join(zarr_dir, 'coloc_pass_cycle', '*.csv')))
files

['/home/datawork-lops-oc/aponte/margot/med_coloc/coloc_pass_cycle/16_502.csv',
 '/home/datawork-lops-oc/aponte/margot/med_coloc/coloc_pass_cycle/16_503.csv',
 '/home/datawork-lops-oc/aponte/margot/med_coloc/coloc_pass_cycle/16_504.csv',
 '/home/datawork-lops-oc/aponte/margot/med_coloc/coloc_pass_cycle/16_505.csv',
 '/home/datawork-lops-oc/aponte/margot/med_coloc/coloc_pass_cycle/16_506.csv',
 '/home/datawork-lops-oc/aponte/margot/med_coloc/coloc_pass_cycle/16_507.csv',
 '/home/datawork-lops-oc/aponte/margot/med_coloc/coloc_pass_cycle/16_509.csv',
 '/home/datawork-lops-oc/aponte/margot/med_coloc/coloc_pass_cycle/16_510.csv',
 '/home/datawork-lops-oc/aponte/margot/med_coloc/coloc_pass_cycle/16_511.csv',
 '/home/datawork-lops-oc/aponte/margot/med_coloc/coloc_pass_cycle/16_512.csv',
 '/home/datawork-lops-oc/aponte/margot/med_coloc/coloc_pass_cycle/16_514.csv',
 '/home/datawork-lops-oc/aponte/margot/med_coloc/coloc_pass_cycle/16_515.csv',
 '/home/datawork-lops-oc/aponte/margot/med_coloc/col

__________
# Editing_flag

In [8]:
cycle = 478
swath = 3

def get_editing_flag(lon, lat, dss):
    R= 6378e3
    diag = 250# * np.sqrt(2)
    dlat = diag*360/(2*np.pi*R)
    dlon = diag*360/(2*np.pi*R*np.cos(lat*np.pi/180))
    testlon = (dss.longitude > lon-dlon) & (dss.longitude < lon+dlon)
    testlat = (dss.latitude > lat-dlat) & (dss.latitude < lat+dlat)
    l = dss.where(testlon &testlat, drop=True).duacs_editing_flag.fillna(int(200)).values
    return '_'.join(l.reshape((1, np.size(l))).astype(int).astype(str)[0])

def apply_get_editing_flag_one(df_, dss):
    l = get_editing_flag(df_.longitude, df_.latitude, dss)
    return l

def apply_get_editing_flag_group(df):
    pass_number = int(df.pass_number.mean())
    cycle_number = int(df.cycle_number.mean())
    try : 
        fi = dfs.where((dfs.pass_number==pass_number)&(dfs.cycle_number==cycle_number)).dropna().file.values[0]
        dss = xr.open_dataset(fi)[['duacs_editing_flag']]
        assert len(fi) !=0, 'empty dss'
    except : 
        assert False, (pass_number, cycle_number)
    
    dfe = df.apply(apply_get_editing_flag_one, args=(dss,), axis=1)
    return dfe.rename('editing_flag')
    

l = apply_get_editing_flag_group(pd.read_csv(files[0]).set_index('row_number').iloc[0:100])

/home1/datahome/mdemol/.miniconda3/envs/histenv/lib/python3.9/site-packages/xarray/backends/plugins.py:110: RuntimeWarning: 'netcdf4' fails while guessing
  warnings.warn(f"{engine!r} fails while guessing", RuntimeWarning)
/home1/datahome/mdemol/.miniconda3/envs/histenv/lib/python3.9/site-packages/xarray/backends/plugins.py:110: RuntimeWarning: 'scipy' fails while guessing
  warnings.warn(f"{engine!r} fails while guessing", RuntimeWarning)


In [9]:
l

row_number
246520            0_0_0_0
246521          0_0_0_200
246522            0_0_0_0
246523            0_0_0_0
246524          0_0_0_200
               ...       
246605      200_0_0_0_0_0
246606            0_0_0_0
246607          0_0_0_200
246608            0_0_0_0
246609    200_0_0_0_200_0
Name: editing_flag, Length: 90, dtype: object

In [ ]:
def editing_flag_all():
    for f in files : 
        f.split('/')[-1]
        df_ = dd.read_csv(f).set_index('row_number').repartition(npartitions=50).persist()
        df_ = df_.map_partitions(apply_get_editing_flag_group, meta=l).compute()
        df_.to_csv(f.replace('coloc_pass_cycle', 'editing_flags'))
        print(f)
    
editing_flag_all()

/home/datawork-lops-oc/aponte/margot/med_coloc/coloc_pass_cycle/16_502.csv
/home/datawork-lops-oc/aponte/margot/med_coloc/coloc_pass_cycle/16_503.csv
/home/datawork-lops-oc/aponte/margot/med_coloc/coloc_pass_cycle/16_504.csv


In [30]:
df = pd.read_csv(files[0].replace('coloc_pass_cycle', 'editing_flags'))

____________
# Locally

In [2]:
files = glob(os.path.join(zarr_dir, 'editing_flags', '*.csv'))

In [6]:
dfs = [pd.read_csv(f).set_index('row_number') for f in files]
df = pd.concat(dfs).sort_index()

In [5]:
legend = {'0' : 'good',
          '5' : 'local_outliers', 
          '10' : 'bad_quality_coast',
          '20' : 'ice',
          '30' : 'soft_outliers',  
          '50' : 'extremes',
          '70' : 'mission_events', 
          '100' : 'bad_swath_extremities', 
          '101' : 'not_on_sea', 
          '102' : 'no_data',
          '200' : 'gradient_nan', 
         }
legend_inv = {legend[k]:k for k in legend}

def count_editing_flags(ed_str, key):
    l = ed_str.split('_')
    return l.count(key)

for key in legend_inv : 
    dfef[key] = dfef.editing_flag.dropna().apply(count_editing_flags, key=legend_inv[key])


In [12]:
df.to_csv('/Users/mdemol/DATA_MED_COLOC/editing_flags_250m.csv')

In [ ]:
legend = {'0' : 'good',
          '5' : 'local_outliers', 
          '10' : 'bad_quality_coast',
          '20' : 'ice',
          '30' : 'soft_outliers',  
          '50' : 'extremes',
          '70' : 'mission_events', 
          '100' : 'bad_swath_extremities', 
          '101' : 'not_on_sea', 
          '102' : 'no_data',
          '200' : 'gradient_nan', 
         }
legend_inv = {legend[k]:k for k in legend}

def count_editing_flags(ed_str, key):
    l = ed_str.split('_')
    return l.count(key)

for key in legend_inv : 
    dfef[key] = dfef.editing_flag.dropna().apply(count_editing_flags, key=legend_inv[key])
